# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B",
]

GEMMA3_SMALL_MODELS = {
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
}

LAMBDA_L1_VALUES = [0.0]

OUTPUT_DIR = "paper_visuals/global_feature_steering"
TABLE_DIR = "paper_tables/global_feature_steering"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

TARGET_DATASETS = [
    "main_val",
    "xed_en",
    "emoevent_en",
]

MODEL_LABELS = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
    "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Path helpers
# ============================================================

def model_safe_name(model_name):
    return model_name.replace("/", "-")


def get_sae_size(model_name):
    if model_name in GEMMA3_SMALL_MODELS:
        return "small"
    return None


def get_feature_dir(model_name, sae_size):
    slug = model_safe_name(model_name)
    if sae_size == "small":
        return f"data/{slug}-feature-act-small"
    return f"data/{slug}-feature-act"


def get_output_dir(model_name, sae_size, lambda_l1):
    feature_dir = get_feature_dir(model_name, sae_size)
    l1_tag = str(lambda_l1).replace(".", "p")

    # This matches your pasted code exactly.
    return os.path.join(
        feature_dir,
        "global_feature_steering",
        f"lr_5e-3_l1_{l1_tag}",
    )


# ============================================================
# Robust column helpers
# ============================================================

def find_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"None of these columns found: {candidates}\nAvailable columns:\n{list(df.columns)}"
        )
    return None


def normalize_dataset_name(x):
    x = str(x)

    mapping = {
        "main_val": "main_val",
        "val": "main_val",
        "validation": "main_val",
        "main_full": "main_full",
        "train": "train",
        "xed_en": "xed_en",
        "emoevent_en": "emoevent_en",
    }

    return mapping.get(x, x)


# ============================================================
# Load all epoch metrics
# ============================================================

def load_all_epoch_metrics():
    dfs = []

    for model_name in MODELS:
        sae_size = get_sae_size(model_name)

        for lambda_l1 in LAMBDA_L1_VALUES:
            output_dir = get_output_dir(model_name, sae_size, lambda_l1)
            metrics_path = os.path.join(output_dir, "epoch_metrics_long.csv")
            summary_path = os.path.join(output_dir, "summary.json")

            if not os.path.exists(metrics_path):
                print(f"[Missing] {metrics_path}")
                continue

            df = pd.read_csv(metrics_path)

            df["model_name"] = model_name
            df["model_label"] = MODEL_LABELS.get(model_name, model_name)
            df["sae_size"] = sae_size if sae_size is not None else "none"
            df["lambda_l1"] = lambda_l1
            df["output_dir"] = output_dir
            df["metrics_path"] = metrics_path

            if os.path.exists(summary_path):
                try:
                    with open(summary_path, "r") as f:
                        summary = json.load(f)
                    df["summary_path"] = summary_path
                except Exception:
                    df["summary_path"] = summary_path
            else:
                df["summary_path"] = None

            dfs.append(df)

    if not dfs:
        raise FileNotFoundError("No epoch_metrics_long.csv files found.")

    all_df = pd.concat(dfs, ignore_index=True)

    epoch_col = find_col(all_df, ["epoch", "epoch_idx", "step", "global_step"])
    dataset_col = find_col(all_df, ["dataset", "dataset_name", "split", "eval_name"])
    macro_f1_col = find_col(
        all_df,
        ["macro_f1", "after_macro_f1", "eval_macro_f1", "f1_macro"],
    )

    all_df = all_df.rename(
        columns={
            epoch_col: "epoch",
            dataset_col: "dataset",
            macro_f1_col: "macro_f1",
        }
    )

    all_df["dataset"] = all_df["dataset"].apply(normalize_dataset_name)
    all_df["epoch"] = pd.to_numeric(all_df["epoch"], errors="coerce")
    all_df["macro_f1"] = pd.to_numeric(all_df["macro_f1"], errors="coerce")

    # Optional columns
    for old, new in [
        ("accuracy", "accuracy"),
        ("acc", "accuracy"),
        ("weighted_f1", "weighted_f1"),
        ("f1_weighted", "weighted_f1"),
        ("ppl", "ppl"),
        ("perplexity", "ppl"),
        ("kl", "kl"),
        ("l1", "l1"),
    ]:
        if old in all_df.columns and new not in all_df.columns:
            all_df[new] = pd.to_numeric(all_df[old], errors="coerce")

    all_df = all_df.dropna(subset=["epoch", "dataset", "macro_f1"])

    return all_df


metrics_df = load_all_epoch_metrics()

print("Loaded metrics:")
print(metrics_df[["model_label", "dataset", "epoch", "macro_f1"]].head())
print("\nDatasets found:", sorted(metrics_df["dataset"].unique()))


# ============================================================
# Best-performance table
# ============================================================

def make_best_result_table(metrics_df):
    df = metrics_df[metrics_df["dataset"].isin(TARGET_DATASETS)].copy()

    idx = (
        df.groupby(["model_name", "model_label", "lambda_l1", "dataset"])["macro_f1"]
        .idxmax()
        .dropna()
    )

    best = df.loc[idx].copy()

    keep_cols = [
        "model_name",
        "model_label",
        "lambda_l1",
        "dataset",
        "epoch",
        "macro_f1",
    ]

    optional_cols = [
        "accuracy",
        "weighted_f1",
        "ppl",
        "kl",
        "l1",
    ]

    keep_cols += [c for c in optional_cols if c in best.columns]

    best = best[keep_cols].sort_values(
        ["model_label", "lambda_l1", "dataset"]
    )

    wide = best.pivot_table(
        index=["model_name", "model_label", "lambda_l1"],
        columns="dataset",
        values=["macro_f1", "epoch"],
        aggfunc="first",
    )

    wide.columns = [f"{metric}_{dataset}" for metric, dataset in wide.columns]
    wide = wide.reset_index()

    ordered_cols = ["model_name", "model_label", "lambda_l1"]

    for dataset in TARGET_DATASETS:
        if f"macro_f1_{dataset}" in wide.columns:
            ordered_cols.append(f"macro_f1_{dataset}")
        if f"epoch_{dataset}" in wide.columns:
            ordered_cols.append(f"epoch_{dataset}")

    wide = wide[ordered_cols]

    best.to_csv(
        os.path.join(TABLE_DIR, "best_epoch_results_long.csv"),
        index=False,
    )

    wide.to_csv(
        os.path.join(TABLE_DIR, "best_epoch_results_wide.csv"),
        index=False,
    )

    # LaTeX table
    latex_df = wide.copy()

    for col in latex_df.columns:
        if col.startswith("macro_f1"):
            latex_df[col] = latex_df[col].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
        if col.startswith("epoch"):
            latex_df[col] = latex_df[col].map(lambda x: f"{int(x)}" if pd.notna(x) else "")

    latex_path = os.path.join(TABLE_DIR, "best_epoch_results_wide.tex")
    with open(latex_path, "w") as f:
        f.write(
            latex_df.to_latex(
                index=False,
                escape=False,
            )
        )

    return best, wide


best_long_df, best_wide_df = make_best_result_table(metrics_df)

print("\nBest results, wide:")
print(best_wide_df)


# ============================================================
# Plots
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_macro_f1_by_epoch_per_dataset(metrics_df):
    df = metrics_df[metrics_df["dataset"].isin(TARGET_DATASETS)].copy()

    for dataset in TARGET_DATASETS:
        sub = df[df["dataset"] == dataset].copy()

        if sub.empty:
            print(f"[Skip] No data for {dataset}")
            continue

        plt.figure(figsize=(7.5, 4.8))

        sns.lineplot(
            data=sub,
            x="epoch",
            y="macro_f1",
            hue="model_label",
            marker="o",
            linewidth=2,
            errorbar=None,
        )

        plt.xlabel("Epoch")
        plt.ylabel("Macro-F1")
        plt.title(dataset)
        plt.legend(
            title="Model",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
        )

        savefig(os.path.join(OUTPUT_DIR, f"macro_f1_by_epoch_{dataset}.png"))


def plot_best_macro_f1_bar(best_long_df):
    df = best_long_df.copy()
    df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

    plt.figure(figsize=(9.5, 4.8))

    sns.barplot(
        data=df,
        x="dataset",
        y="macro_f1",
        hue="model_label",
    )

    plt.xlabel("Dataset")
    plt.ylabel("Best Macro-F1")
    plt.title("Best validation/external performance")
    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    savefig(os.path.join(OUTPUT_DIR, "best_macro_f1_all_datasets.png"))


def plot_best_epoch_bar(best_long_df):
    df = best_long_df.copy()
    df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

    plt.figure(figsize=(9.5, 4.8))

    sns.barplot(
        data=df,
        x="dataset",
        y="epoch",
        hue="model_label",
    )

    plt.xlabel("Dataset")
    plt.ylabel("Best epoch")
    plt.title("Epoch selected by best Macro-F1")
    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    savefig(os.path.join(OUTPUT_DIR, "best_epoch_all_datasets.png"))


def plot_main_val_vs_external(best_wide_df):
    df = best_wide_df.copy()

    if "macro_f1_main_val" not in df.columns:
        print("[Skip] No main_val results.")
        return

    external_cols = [
        c for c in [
            "macro_f1_xed_en",
            "macro_f1_emoevent_en",
        ]
        if c in df.columns
    ]

    if not external_cols:
        print("[Skip] No external dataset results.")
        return

    df["external_mean_macro_f1"] = df[external_cols].mean(axis=1)

    plt.figure(figsize=(5.8, 4.8))

    sns.scatterplot(
        data=df,
        x="macro_f1_main_val",
        y="external_mean_macro_f1",
        hue="model_label",
        s=80,
    )

    for _, row in df.iterrows():
        plt.text(
            row["macro_f1_main_val"] + 0.002,
            row["external_mean_macro_f1"] + 0.002,
            row["model_label"],
            fontsize=8,
        )

    plt.xlabel("Best Macro-F1 on main val")
    plt.ylabel("Mean best Macro-F1 on external datasets")
    plt.title("Main validation vs. external generalization")
    plt.legend([], [], frameon=False)

    savefig(os.path.join(OUTPUT_DIR, "main_val_vs_external_mean_macro_f1.png"))


plot_macro_f1_by_epoch_per_dataset(metrics_df)
plot_best_macro_f1_bar(best_long_df)
plot_best_epoch_bar(best_long_df)
plot_main_val_vs_external(best_wide_df)

print("\nSaved plots to:", OUTPUT_DIR)
print("Saved tables to:", TABLE_DIR)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

TABLE_DIR = "paper_tables/global_feature_steering"
os.makedirs(TABLE_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

DATASET_ORDER = ["main_val", "emoevent_en", "xed_en"]
DATASET_LABELS = {
    "main_val": "main",
    "emoevent_en": "emoevent",
    "xed_en": "xed",
}

MODEL_LABELS = {
    "google-gemma-2-2b": "Gemma-2-2B",
    "google-gemma-2-9b": "Gemma-2-9B",
    "google-gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google-gemma-3-4b-it": "Gemma-3-4B-IT",
    "google-gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google-gemma-3-12b-it": "Gemma-3-12B-IT",
    "meta-llama-Llama-3.1-8B": "Llama-3.1-8B",
}

def infer_model_from_path(path):
    parts = path.split(os.sep)
    for p in parts:
        if p.endswith("-feature-act") or p.endswith("-feature-act-small"):
            return p.replace("-feature-act-small", "").replace("-feature-act", "")
    return None

def normalize_dataset(x):
    return {
        "val": "main_val",
        "validation": "main_val",
        "xed": "xed_en",
        "emoevent": "emoevent_en",
    }.get(str(x), str(x))

def load_all_steering_outputs():
    metric_paths = glob.glob(
        "data/*feature-act*/global_feature_steering/lr_*/epoch_metrics_long.csv"
    )

    metric_dfs = []
    per_emotion_dfs = []

    for metrics_path in sorted(metric_paths):
        run_dir = os.path.dirname(metrics_path)
        per_emotion_path = os.path.join(run_dir, "epoch_per_emotion_f1.csv")

        model_key = infer_model_from_path(metrics_path)
        model_label = MODEL_LABELS.get(model_key, model_key)

        metrics = pd.read_csv(metrics_path)
        metrics["model_key"] = model_key
        metrics["model"] = model_label
        metrics["run_dir"] = run_dir
        metrics["dataset"] = metrics["dataset"].apply(normalize_dataset)

        metric_dfs.append(metrics)

        if os.path.exists(per_emotion_path):
            pe = pd.read_csv(per_emotion_path)
            pe["model_key"] = model_key
            pe["model"] = model_label
            pe["run_dir"] = run_dir
            pe["dataset"] = pe["dataset"].apply(normalize_dataset)
            per_emotion_dfs.append(pe)
        else:
            print(f"[Missing per-emotion file] {per_emotion_path}")

    if not metric_dfs:
        raise FileNotFoundError("No epoch_metrics_long.csv files found.")

    metrics_df = pd.concat(metric_dfs, ignore_index=True)

    if not per_emotion_dfs:
        raise FileNotFoundError("No epoch_per_emotion_f1.csv files found.")

    per_emotion_df = pd.concat(per_emotion_dfs, ignore_index=True)

    return metrics_df, per_emotion_df

metrics_df, per_emotion_df = load_all_steering_outputs()

# ============================================================
# Select best epoch by main_val macro-F1
# ============================================================

main_val = metrics_df[metrics_df["dataset"] == "main_val"].copy()

best_idx = (
    main_val.groupby(["model_key", "run_dir"])["macro_f1"]
    .idxmax()
    .dropna()
)

best_epoch_df = main_val.loc[best_idx, ["model_key", "run_dir", "epoch"]]
best_epoch_df = best_epoch_df.rename(columns={"epoch": "best_epoch"})

# If duplicate runs exist for the same model, keep the best run.
best_run_idx = (
    main_val.loc[best_idx]
    .groupby("model_key")["macro_f1"]
    .idxmax()
    .dropna()
)

best_runs = main_val.loc[best_run_idx, ["model_key", "model", "run_dir", "epoch", "macro_f1"]]
best_runs = best_runs.rename(columns={"epoch": "best_epoch"})

# ============================================================
# Build final paper table
# ============================================================

FINAL_DATASETS = [
    "main_val",
    "emoevent_en",
    "xed_en",
]

FINAL_DATASET_LABELS = {
    "main_val": "main",
    "emoevent_en": "emoevent",
    "xed_en": "xed",
}

rows = []

for _, br in best_runs.iterrows():

    model_key = br["model_key"]
    model = br["model"]
    run_dir = br["run_dir"]
    best_epoch = int(br["best_epoch"])

    for stage_name, epoch in [
        ("pretrained", 0),
        ("steered", best_epoch),
    ]:

        row = {
            "model": model,
            "setting": stage_name,
        }

        m_sub = metrics_df[
            (metrics_df["model_key"] == model_key)
            & (metrics_df["run_dir"] == run_dir)
            & (metrics_df["epoch"] == epoch)
        ].copy()

        pe_sub = per_emotion_df[
            (per_emotion_df["model_key"] == model_key)
            & (per_emotion_df["run_dir"] == run_dir)
            & (per_emotion_df["epoch"] == epoch)
        ].copy()

        # ----------------------------------------------------
        # Main dataset:
        # macro + per-emotion + ppl
        # ----------------------------------------------------

        main_row = m_sub[m_sub["dataset"] == "main_val"]

        if len(main_row) > 0:

            main_row = main_row.iloc[0]

            row["main_macro_f1"] = main_row["macro_f1"]

            if "general_perplexity" in main_row.index:
                row["ppl"] = main_row["general_perplexity"]

            main_pe = pe_sub[pe_sub["dataset"] == "main_val"]

            for emotion in EMOTIONS:

                e_row = main_pe[main_pe["emotion"] == emotion]

                if len(e_row) > 0:
                    row[f"{emotion}_f1"] = e_row.iloc[0]["f1"]

        # ----------------------------------------------------
        # External datasets:
        # ONLY macro F1
        # ----------------------------------------------------

        for dataset in ["emoevent_en", "xed_en"]:

            dlabel = FINAL_DATASET_LABELS[dataset]

            d_row = m_sub[m_sub["dataset"] == dataset]

            if len(d_row) > 0:

                d_row = d_row.iloc[0]

                row[f"{dlabel}_macro_f1"] = d_row["macro_f1"]

        rows.append(row)

final_df = pd.DataFrame(rows)

# ============================================================
# Column ordering
# ============================================================

ordered_cols = [
    "model",
    "setting",

    "main_macro_f1",

    "anger_f1",
    "disgust_f1",
    "fear_f1",
    "joy_f1",
    "sadness_f1",
    "surprise_f1",

    "emoevent_macro_f1",
    "xed_macro_f1",

    "ppl",
]

ordered_cols = [c for c in ordered_cols if c in final_df.columns]

final_df = final_df[ordered_cols]

# ============================================================
# Round
# ============================================================

for col in final_df.columns:

    if col in ["model", "setting"]:
        continue

    final_df[col] = (
        pd.to_numeric(final_df[col], errors="coerce")
        .round(3)
    )

# ============================================================
# Sort
# ============================================================

final_df["setting_order"] = final_df["setting"].map({
    "pretrained": 0,
    "steered": 1,
})

final_df = final_df.sort_values(
    ["model", "setting_order"]
).drop(columns=["setting_order"])

# ============================================================
# Save
# ============================================================

csv_path = os.path.join(
    TABLE_DIR,
    "final_paper_table.csv",
)

tex_path = os.path.join(
    TABLE_DIR,
    "final_paper_table.tex",
)

final_df.to_csv(csv_path, index=False)

with open(tex_path, "w") as f:
    f.write(
        final_df.to_latex(
            index=False,
            escape=False,
        )
    )

print(final_df)

print("\nSaved:")
print(csv_path)
print(tex_path)

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B",
]

GEMMA3_SMALL_MODELS = {
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
}

LAMBDA_L1_VALUES = [0.001]

OUTPUT_DIR = "paper_visuals/global_feature_steering"
TABLE_DIR = "paper_tables/global_feature_steering"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

TARGET_DATASETS = [
    "main_val",
    "xed_en",
    "emoevent_en",
]

MODEL_LABELS = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
    "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Path helpers
# ============================================================

def model_safe_name(model_name):
    return model_name.replace("/", "-")


def get_sae_size(model_name):
    if model_name in GEMMA3_SMALL_MODELS:
        return "small"
    return None


def get_feature_dir(model_name, sae_size):
    slug = model_safe_name(model_name)
    if sae_size == "small":
        return f"data/{slug}-feature-act-small"
    return f"data/{slug}-feature-act"


def get_output_dir(model_name, sae_size, lambda_l1):
    feature_dir = get_feature_dir(model_name, sae_size)
    l1_tag = str(lambda_l1).replace(".", "p")

    # This matches your pasted code exactly.
    return os.path.join(
        feature_dir,
        "global_feature_steering",
        f"lr_5e-3_l1_{l1_tag}",
    )


# ============================================================
# Robust column helpers
# ============================================================

def find_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"None of these columns found: {candidates}\nAvailable columns:\n{list(df.columns)}"
        )
    return None


def normalize_dataset_name(x):
    x = str(x)

    mapping = {
        "main_val": "main_val",
        "val": "main_val",
        "validation": "main_val",
        "main_full": "main_full",
        "train": "train",
        "xed_en": "xed_en",
        "emoevent_en": "emoevent_en",
    }

    return mapping.get(x, x)


# ============================================================
# Load all epoch metrics
# ============================================================

def load_all_epoch_metrics():
    dfs = []

    for model_name in MODELS:
        sae_size = get_sae_size(model_name)

        for lambda_l1 in LAMBDA_L1_VALUES:
            output_dir = get_output_dir(model_name, sae_size, lambda_l1)
            metrics_path = os.path.join(output_dir, "epoch_metrics_long.csv")
            summary_path = os.path.join(output_dir, "summary.json")

            if not os.path.exists(metrics_path):
                print(f"[Missing] {metrics_path}")
                continue

            df = pd.read_csv(metrics_path)

            df["model_name"] = model_name
            df["model_label"] = MODEL_LABELS.get(model_name, model_name)
            df["sae_size"] = sae_size if sae_size is not None else "none"
            df["lambda_l1"] = lambda_l1
            df["output_dir"] = output_dir
            df["metrics_path"] = metrics_path

            if os.path.exists(summary_path):
                try:
                    with open(summary_path, "r") as f:
                        summary = json.load(f)
                    df["summary_path"] = summary_path
                except Exception:
                    df["summary_path"] = summary_path
            else:
                df["summary_path"] = None

            dfs.append(df)

    if not dfs:
        raise FileNotFoundError("No epoch_metrics_long.csv files found.")

    all_df = pd.concat(dfs, ignore_index=True)

    epoch_col = find_col(all_df, ["epoch", "epoch_idx", "step", "global_step"])
    dataset_col = find_col(all_df, ["dataset", "dataset_name", "split", "eval_name"])
    macro_f1_col = find_col(
        all_df,
        ["macro_f1", "after_macro_f1", "eval_macro_f1", "f1_macro"],
    )

    all_df = all_df.rename(
        columns={
            epoch_col: "epoch",
            dataset_col: "dataset",
            macro_f1_col: "macro_f1",
        }
    )

    all_df["dataset"] = all_df["dataset"].apply(normalize_dataset_name)
    all_df["epoch"] = pd.to_numeric(all_df["epoch"], errors="coerce")
    all_df["macro_f1"] = pd.to_numeric(all_df["macro_f1"], errors="coerce")

    # Optional columns
    for old, new in [
        ("accuracy", "accuracy"),
        ("acc", "accuracy"),
        ("weighted_f1", "weighted_f1"),
        ("f1_weighted", "weighted_f1"),
        ("ppl", "ppl"),
        ("perplexity", "ppl"),
        ("kl", "kl"),
        ("l1", "l1"),
    ]:
        if old in all_df.columns and new not in all_df.columns:
            all_df[new] = pd.to_numeric(all_df[old], errors="coerce")

    all_df = all_df.dropna(subset=["epoch", "dataset", "macro_f1"])

    return all_df


metrics_df = load_all_epoch_metrics()

print("Loaded metrics:")
print(metrics_df[["model_label", "dataset", "epoch", "macro_f1"]].head())
print("\nDatasets found:", sorted(metrics_df["dataset"].unique()))


# ============================================================
# Best-performance table
# ============================================================

def make_best_result_table(metrics_df):
    df = metrics_df[metrics_df["dataset"].isin(TARGET_DATASETS)].copy()

    idx = (
        df.groupby(["model_name", "model_label", "lambda_l1", "dataset"])["macro_f1"]
        .idxmax()
        .dropna()
    )

    best = df.loc[idx].copy()

    keep_cols = [
        "model_name",
        "model_label",
        "lambda_l1",
        "dataset",
        "epoch",
        "macro_f1",
    ]

    optional_cols = [
        "accuracy",
        "weighted_f1",
        "ppl",
        "kl",
        "l1",
    ]

    keep_cols += [c for c in optional_cols if c in best.columns]

    best = best[keep_cols].sort_values(
        ["model_label", "lambda_l1", "dataset"]
    )

    wide = best.pivot_table(
        index=["model_name", "model_label", "lambda_l1"],
        columns="dataset",
        values=["macro_f1", "epoch"],
        aggfunc="first",
    )

    wide.columns = [f"{metric}_{dataset}" for metric, dataset in wide.columns]
    wide = wide.reset_index()

    ordered_cols = ["model_name", "model_label", "lambda_l1"]

    for dataset in TARGET_DATASETS:
        if f"macro_f1_{dataset}" in wide.columns:
            ordered_cols.append(f"macro_f1_{dataset}")
        if f"epoch_{dataset}" in wide.columns:
            ordered_cols.append(f"epoch_{dataset}")

    wide = wide[ordered_cols]

    best.to_csv(
        os.path.join(TABLE_DIR, "best_epoch_results_long.csv"),
        index=False,
    )

    wide.to_csv(
        os.path.join(TABLE_DIR, "best_epoch_results_wide.csv"),
        index=False,
    )

    # LaTeX table
    latex_df = wide.copy()

    for col in latex_df.columns:
        if col.startswith("macro_f1"):
            latex_df[col] = latex_df[col].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
        if col.startswith("epoch"):
            latex_df[col] = latex_df[col].map(lambda x: f"{int(x)}" if pd.notna(x) else "")

    latex_path = os.path.join(TABLE_DIR, "best_epoch_results_wide.tex")
    with open(latex_path, "w") as f:
        f.write(
            latex_df.to_latex(
                index=False,
                escape=False,
            )
        )

    return best, wide


best_long_df, best_wide_df = make_best_result_table(metrics_df)

print("\nBest results, wide:")
print(best_wide_df)


# ============================================================
# Plots
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_macro_f1_by_epoch_per_dataset(metrics_df):
    df = metrics_df[metrics_df["dataset"].isin(TARGET_DATASETS)].copy()

    for dataset in TARGET_DATASETS:
        sub = df[df["dataset"] == dataset].copy()

        if sub.empty:
            print(f"[Skip] No data for {dataset}")
            continue

        plt.figure(figsize=(7.5, 4.8))

        sns.lineplot(
            data=sub,
            x="epoch",
            y="macro_f1",
            hue="model_label",
            marker="o",
            linewidth=2,
            errorbar=None,
        )

        plt.xlabel("Epoch")
        plt.ylabel("Macro-F1")
        plt.title(dataset)
        plt.legend(
            title="Model",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
        )

        savefig(os.path.join(OUTPUT_DIR, f"macro_f1_by_epoch_{dataset}.png"))


def plot_best_macro_f1_bar(best_long_df):
    df = best_long_df.copy()
    df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

    plt.figure(figsize=(9.5, 4.8))

    sns.barplot(
        data=df,
        x="dataset",
        y="macro_f1",
        hue="model_label",
    )

    plt.xlabel("Dataset")
    plt.ylabel("Best Macro-F1")
    plt.title("Best validation/external performance")
    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    savefig(os.path.join(OUTPUT_DIR, "best_macro_f1_all_datasets.png"))


def plot_best_epoch_bar(best_long_df):
    df = best_long_df.copy()
    df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

    plt.figure(figsize=(9.5, 4.8))

    sns.barplot(
        data=df,
        x="dataset",
        y="epoch",
        hue="model_label",
    )

    plt.xlabel("Dataset")
    plt.ylabel("Best epoch")
    plt.title("Epoch selected by best Macro-F1")
    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    savefig(os.path.join(OUTPUT_DIR, "best_epoch_all_datasets.png"))


def plot_main_val_vs_external(best_wide_df):
    df = best_wide_df.copy()

    if "macro_f1_main_val" not in df.columns:
        print("[Skip] No main_val results.")
        return

    external_cols = [
        c for c in [
            "macro_f1_xed_en",
            "macro_f1_emoevent_en",
        ]
        if c in df.columns
    ]

    if not external_cols:
        print("[Skip] No external dataset results.")
        return

    df["external_mean_macro_f1"] = df[external_cols].mean(axis=1)

    plt.figure(figsize=(5.8, 4.8))

    sns.scatterplot(
        data=df,
        x="macro_f1_main_val",
        y="external_mean_macro_f1",
        hue="model_label",
        s=80,
    )

    for _, row in df.iterrows():
        plt.text(
            row["macro_f1_main_val"] + 0.002,
            row["external_mean_macro_f1"] + 0.002,
            row["model_label"],
            fontsize=8,
        )

    plt.xlabel("Best Macro-F1 on main val")
    plt.ylabel("Mean best Macro-F1 on external datasets")
    plt.title("Main validation vs. external generalization")
    plt.legend([], [], frameon=False)

    savefig(os.path.join(OUTPUT_DIR, "main_val_vs_external_mean_macro_f1.png"))


plot_macro_f1_by_epoch_per_dataset(metrics_df)
plot_best_macro_f1_bar(best_long_df)
plot_best_epoch_bar(best_long_df)
plot_main_val_vs_external(best_wide_df)

print("\nSaved plots to:", OUTPUT_DIR)
print("Saved tables to:", TABLE_DIR)

In [ ]:
import os
import glob
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

BASE_DIRS = [
    "data/google-gemma-2-2b-feature-act/global_feature_steering",
    "data/google-gemma-2-9b-feature-act/global_feature_steering",
    "data/google-gemma-3-4b-pt-feature-act-small/global_feature_steering",
    "data/google-gemma-3-4b-it-feature-act-small/global_feature_steering",
    "data/google-gemma-3-12b-pt-feature-act-small/global_feature_steering",
    "data/google-gemma-3-12b-it-feature-act-small/global_feature_steering",
]

OUTPUT_DIR = "paper_visuals/global_feature_steering_multiplier_analysis"
TABLE_DIR = "paper_tables/global_feature_steering_multiplier_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

MODEL_LABELS = {
    "google-gemma-2-2b-feature-act": "Gemma-2-2B",
    "google-gemma-2-9b-feature-act": "Gemma-2-9B",
    "google-gemma-3-4b-pt-feature-act-small": "Gemma-3-4B-PT",
    "google-gemma-3-4b-it-feature-act-small": "Gemma-3-4B-IT",
    "google-gemma-3-12b-pt-feature-act-small": "Gemma-3-12B-PT",
    "google-gemma-3-12b-it-feature-act-small": "Gemma-3-12B-IT",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Helpers
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def parse_feature_emotions(x):
    """
    Parse feature emotion field.
    Supports:
    - stringified list
    - comma separated
    - single label
    """

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    x = str(x).strip()

    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v).strip().lower() for v in parsed]
    except Exception:
        pass

    if "," in x:
        return [v.strip().lower() for v in x.split(",")]

    return [x.lower()]


# ============================================================
# Load learned multipliers
# ============================================================

all_rows = []

for base_dir in BASE_DIRS:

    run_dirs = sorted(glob.glob(os.path.join(base_dir, "lr_*")))

    for run_dir in run_dirs:

        mult_path = os.path.join(run_dir, "learned_feature_multipliers.csv")

        if not os.path.exists(mult_path):
            print(f"[Missing] {mult_path}")
            continue

        df = pd.read_csv(mult_path)

        model_key = None

        for k in MODEL_LABELS:
            if k in run_dir:
                model_key = k
                break

        df["model_key"] = model_key
        df["model_label"] = MODEL_LABELS.get(model_key, model_key)
        df["run_dir"] = run_dir

        # ----------------------------------------------------
        # Infer multiplier column
        # ----------------------------------------------------

        multiplier_col = None

        for c in [
            "multiplier",
            "learned_multiplier",
            "feature_multiplier",
            "value",
        ]:
            if c in df.columns:
                multiplier_col = c
                break

        if multiplier_col is None:
            raise ValueError(
                f"Cannot find multiplier column in {mult_path}\nColumns: {list(df.columns)}"
            )

        df["multiplier_value"] = pd.to_numeric(df[multiplier_col], errors="coerce")

        # ----------------------------------------------------
        # Infer emotion ownership
        # ----------------------------------------------------

        emotion_col = None

        for c in [
            "affected_emotions",
            "emotion",
            "emotions",
            "target_emotions",
        ]:
            if c in df.columns:
                emotion_col = c
                break

        if emotion_col is None:
            raise ValueError(
                f"Cannot find emotion column in {mult_path}\nColumns: {list(df.columns)}"
            )

        df["emotion_list"] = df[emotion_col].apply(parse_feature_emotions)

        exploded = df.explode("emotion_list").rename(
            columns={"emotion_list": "emotion"}
        )

        exploded["emotion"] = exploded["emotion"].astype(str).str.lower()

        exploded = exploded[exploded["emotion"].isin(EMOTIONS)]

        all_rows.append(exploded)

if not all_rows:
    raise ValueError("No learned_feature_multipliers.csv files found.")

mult_df = pd.concat(all_rows, ignore_index=True)

print(mult_df.head())


# ============================================================
# Basic statistics
# ============================================================

summary_stats = (
    mult_df.groupby(["model_label", "emotion"])
    .agg(
        mean_multiplier=("multiplier_value", "mean"),
        median_multiplier=("multiplier_value", "median"),
        std_multiplier=("multiplier_value", "std"),
        n_features=("multiplier_value", "size"),
        pct_gt1=("multiplier_value", lambda x: (x > 1.0).mean()),
        pct_lt1=("multiplier_value", lambda x: (x < 1.0).mean()),
    )
    .reset_index()
)

summary_stats.to_csv(
    os.path.join(TABLE_DIR, "multiplier_summary_stats.csv"),
    index=False,
)

print(summary_stats)


# ============================================================
# Detailed analysis plots
# ============================================================

# ------------------------------------------------------------
# Distribution by emotion
# ------------------------------------------------------------

plt.figure(figsize=(8.2, 4.8))

sns.violinplot(
    data=mult_df,
    x="emotion",
    y="multiplier_value",
    inner="box",
    cut=0,
)

plt.axhline(1.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Learned multiplier")
plt.title("Distribution of learned multipliers by emotion")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "multiplier_distribution_by_emotion.png",
    )
)


# ------------------------------------------------------------
# Mean multiplier per emotion across models
# ------------------------------------------------------------

agg = (
    mult_df.groupby(["model_label", "emotion"])
    .agg(
        mean_multiplier=("multiplier_value", "mean"),
        std_multiplier=("multiplier_value", "std"),
    )
    .reset_index()
)

plt.figure(figsize=(8.5, 4.8))

sns.barplot(
    data=agg,
    x="emotion",
    y="mean_multiplier",
    hue="model_label",
)

plt.axhline(1.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Mean learned multiplier")
plt.title("Average learned multiplier by emotion")

plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

savefig(
    os.path.join(
        OUTPUT_DIR,
        "mean_multiplier_by_emotion_and_model.png",
    )
)


# ------------------------------------------------------------
# Amplified vs suppressed feature ratios
# ------------------------------------------------------------

ratio_df = (
    mult_df.groupby(["model_label", "emotion"])
    .agg(
        amplified_ratio=("multiplier_value", lambda x: (x > 1.0).mean()),
        suppressed_ratio=("multiplier_value", lambda x: (x < 1.0).mean()),
    )
    .reset_index()
)

ratio_long = ratio_df.melt(
    id_vars=["model_label", "emotion"],
    value_vars=["amplified_ratio", "suppressed_ratio"],
    var_name="direction",
    value_name="ratio",
)

plt.figure(figsize=(9.2, 4.8))

sns.barplot(
    data=ratio_long,
    x="emotion",
    y="ratio",
    hue="direction",
)

plt.xlabel("Feature emotion")
plt.ylabel("Feature ratio")
plt.title("Amplified vs. suppressed feature ratios")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "amplified_vs_suppressed_feature_ratios.png",
    )
)


# ============================================================
# Comprehensive analysis
# Which emotion features are amplified/suppressed?
# ============================================================

emotion_global = (
    mult_df.groupby("emotion")
    .agg(
        mean_multiplier=("multiplier_value", "mean"),
        median_multiplier=("multiplier_value", "median"),
        amplified_ratio=("multiplier_value", lambda x: (x > 1.0).mean()),
        suppressed_ratio=("multiplier_value", lambda x: (x < 1.0).mean()),
        n_features=("multiplier_value", "size"),
    )
    .reset_index()
)

emotion_global["net_direction"] = (
    emotion_global["amplified_ratio"]
    - emotion_global["suppressed_ratio"]
)

emotion_global = emotion_global.sort_values(
    "mean_multiplier",
    ascending=False,
)

emotion_global.to_csv(
    os.path.join(
        TABLE_DIR,
        "global_emotion_multiplier_summary.csv",
    ),
    index=False,
)

print("\nGlobal emotion multiplier summary:")
print(emotion_global)


# ------------------------------------------------------------
# Global emotion tendency plot
# ------------------------------------------------------------

plt.figure(figsize=(7.5, 4.8))

sns.barplot(
    data=emotion_global,
    x="emotion",
    y="mean_multiplier",
)

plt.axhline(1.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Mean learned multiplier")
plt.title("Global amplification tendency by emotion")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "global_amplification_tendency_by_emotion.png",
    )
)


# ------------------------------------------------------------
# Net amplification score
# ------------------------------------------------------------

plt.figure(figsize=(7.5, 4.8))

sns.barplot(
    data=emotion_global,
    x="emotion",
    y="net_direction",
)

plt.axhline(0.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Amplified ratio − suppressed ratio")
plt.title("Net amplification tendency by emotion")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "net_amplification_tendency_by_emotion.png",
    )
)


# ============================================================
# Save comprehensive table
# ============================================================

emotion_global.to_csv(
    os.path.join(
        TABLE_DIR,
        "emotion_feature_amplification_analysis.csv",
    ),
    index=False,
)

latex_df = emotion_global.copy()

for col in [
    "mean_multiplier",
    "median_multiplier",
    "amplified_ratio",
    "suppressed_ratio",
    "net_direction",
]:
    latex_df[col] = latex_df[col].map(lambda x: f"{x:.3f}")

with open(
    os.path.join(
        TABLE_DIR,
        "emotion_feature_amplification_analysis.tex",
    ),
    "w",
) as f:
    f.write(
        latex_df.to_latex(
            index=False,
            escape=False,
        )
    )

print("\nSaved plots to:", OUTPUT_DIR)
print("Saved tables to:", TABLE_DIR)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# Global Style Alignment
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 12,
})

# Data Setup
plot_df = mult_df.copy()
plot_df["multiplier_plot"] = plot_df["multiplier_value"].clip(0, 10)
emotion_order = [e.capitalize() for e in EMOTIONS]
plot_df["emotion_label"] = plot_df["emotion"].str.capitalize()

fig, ax = plt.subplots(figsize=(8.2, 4.8))

# 1. Background Violin
# Removed showmeans/meanprops to fix the AttributeError
sns.violinplot(
    data=plot_df,
    x="emotion_label",
    y="multiplier_plot",
    order=emotion_order,
    color="#5B84B1",
    inner=None,
    linewidth=1.2,
    alpha=0.6,
    cut=0,
    ax=ax
)

# 2. Add Mean Markers (White circles with black edges)
# We calculate the mean manually and plot it to avoid the AttributeError
means = plot_df.groupby("emotion_label")["multiplier_plot"].mean().reindex(emotion_order)
ax.scatter(
    x=range(len(emotion_order)), 
    y=means, 
    color="white", 
    edgecolor="black", 
    s=40, 
    zorder=4, 
    label="Mean"
)

# 3. Thin boxplot overlay
sns.boxplot(
    data=plot_df,
    x="emotion_label",
    y="multiplier_plot",
    order=emotion_order,
    width=0.18,
    showcaps=True,
    boxprops={"facecolor": "white", "zorder": 3, "alpha": 0.8},
    showfliers=False,
    whiskerprops={"linewidth": 1},
    ax=ax
)

# 4. Light jittered points
sns.stripplot(
    data=plot_df.sample(min(len(plot_df), 3000), random_state=42),
    x="emotion_label",
    y="multiplier_plot",
    order=emotion_order,
    color="black",
    alpha=0.08,
    size=2,
    jitter=0.25,
    ax=ax
)

plt.axhline(1.0, linestyle="--", alpha=0.6, color="black")

# Spines cleanup
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.ylim(0, 2)
plt.xlabel("") 
plt.ylabel("Learned multiplier")
plt.title("Distribution of learned multipliers by emotion", fontsize=12)
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B",
]

GEMMA3_SMALL_MODELS = {
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
}

LAMBDA_L1_VALUES = [0.0]

OUTPUT_DIR = "paper_visuals/global_feature_steering_5shots"
TABLE_DIR = "paper_tables/global_feature_steering_5shots"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

TARGET_DATASETS = [
    "main_val",
    "xed_en",
    "emoevent_en",
]

MODEL_LABELS = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
    "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Path helpers
# ============================================================

def model_safe_name(model_name):
    return model_name.replace("/", "-")


def get_sae_size(model_name):
    if model_name in GEMMA3_SMALL_MODELS:
        return "small"
    return None


def get_feature_dir(model_name, sae_size):
    slug = model_safe_name(model_name)
    if sae_size == "small":
        return f"data/{slug}-feature-act-small"
    return f"data/{slug}-feature-act"


def get_output_dir(model_name, sae_size, lambda_l1):
    feature_dir = get_feature_dir(model_name, sae_size)
    l1_tag = str(lambda_l1).replace(".", "p")

    # This matches your pasted code exactly.
    return os.path.join(
        feature_dir,
        "global_feature_steering_5shot",
        f"steps_1500_eval_300_bs_8_lr_1e-2_l1_{l1_tag}",
    )


# ============================================================
# Robust column helpers
# ============================================================

def find_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(
            f"None of these columns found: {candidates}\nAvailable columns:\n{list(df.columns)}"
        )
    return None


def normalize_dataset_name(x):
    x = str(x)

    mapping = {
        "main_val": "main_val",
        "val": "main_val",
        "validation": "main_val",
        "main_full": "main_full",
        "train": "train",
        "xed_en": "xed_en",
        "emoevent_en": "emoevent_en",
    }

    return mapping.get(x, x)


# ============================================================
# Load all epoch metrics
# ============================================================

def load_all_epoch_metrics():
    dfs = []

    for model_name in MODELS:
        sae_size = get_sae_size(model_name)

        for lambda_l1 in LAMBDA_L1_VALUES:
            output_dir = get_output_dir(model_name, sae_size, lambda_l1)
            metrics_path = os.path.join(output_dir, "epoch_metrics_long.csv")
            summary_path = os.path.join(output_dir, "summary.json")

            if not os.path.exists(metrics_path):
                print(f"[Missing] {metrics_path}")
                continue

            df = pd.read_csv(metrics_path)

            df["model_name"] = model_name
            df["model_label"] = MODEL_LABELS.get(model_name, model_name)
            df["sae_size"] = sae_size if sae_size is not None else "none"
            df["lambda_l1"] = lambda_l1
            df["output_dir"] = output_dir
            df["metrics_path"] = metrics_path

            if os.path.exists(summary_path):
                try:
                    with open(summary_path, "r") as f:
                        summary = json.load(f)
                    df["summary_path"] = summary_path
                except Exception:
                    df["summary_path"] = summary_path
            else:
                df["summary_path"] = None

            dfs.append(df)

    if not dfs:
        raise FileNotFoundError("No epoch_metrics_long.csv files found.")

    all_df = pd.concat(dfs, ignore_index=True)

    epoch_col = find_col(all_df, ["epoch", "epoch_idx", "step", "global_step"])
    dataset_col = find_col(all_df, ["dataset", "dataset_name", "split", "eval_name"])
    macro_f1_col = find_col(
        all_df,
        ["macro_f1", "after_macro_f1", "eval_macro_f1", "f1_macro"],
    )

    all_df = all_df.rename(
        columns={
            epoch_col: "epoch",
            dataset_col: "dataset",
            macro_f1_col: "macro_f1",
        }
    )

    all_df["dataset"] = all_df["dataset"].apply(normalize_dataset_name)
    all_df["epoch"] = pd.to_numeric(all_df["epoch"], errors="coerce")
    all_df["macro_f1"] = pd.to_numeric(all_df["macro_f1"], errors="coerce")

    # Optional columns
    for old, new in [
        ("accuracy", "accuracy"),
        ("acc", "accuracy"),
        ("weighted_f1", "weighted_f1"),
        ("f1_weighted", "weighted_f1"),
        ("ppl", "ppl"),
        ("perplexity", "ppl"),
        ("kl", "kl"),
        ("l1", "l1"),
    ]:
        if old in all_df.columns and new not in all_df.columns:
            all_df[new] = pd.to_numeric(all_df[old], errors="coerce")

    all_df = all_df.dropna(subset=["epoch", "dataset", "macro_f1"])

    return all_df


metrics_df = load_all_epoch_metrics()

print("Loaded metrics:")
print(metrics_df[["model_label", "dataset", "epoch", "macro_f1"]].head())
print("\nDatasets found:", sorted(metrics_df["dataset"].unique()))


# ============================================================
# Best-performance table
# ============================================================

def make_best_result_table(metrics_df):
    df = metrics_df[metrics_df["dataset"].isin(TARGET_DATASETS)].copy()

    idx = (
        df.groupby(["model_name", "model_label", "lambda_l1", "dataset"])["macro_f1"]
        .idxmax()
        .dropna()
    )

    best = df.loc[idx].copy()

    keep_cols = [
        "model_name",
        "model_label",
        "lambda_l1",
        "dataset",
        "epoch",
        "macro_f1",
    ]

    optional_cols = [
        "accuracy",
        "weighted_f1",
        "ppl",
        "kl",
        "l1",
    ]

    keep_cols += [c for c in optional_cols if c in best.columns]

    best = best[keep_cols].sort_values(
        ["model_label", "lambda_l1", "dataset"]
    )

    wide = best.pivot_table(
        index=["model_name", "model_label", "lambda_l1"],
        columns="dataset",
        values=["macro_f1", "epoch"],
        aggfunc="first",
    )

    wide.columns = [f"{metric}_{dataset}" for metric, dataset in wide.columns]
    wide = wide.reset_index()

    ordered_cols = ["model_name", "model_label", "lambda_l1"]

    for dataset in TARGET_DATASETS:
        if f"macro_f1_{dataset}" in wide.columns:
            ordered_cols.append(f"macro_f1_{dataset}")
        if f"epoch_{dataset}" in wide.columns:
            ordered_cols.append(f"epoch_{dataset}")

    wide = wide[ordered_cols]

    best.to_csv(
        os.path.join(TABLE_DIR, "best_epoch_results_long.csv"),
        index=False,
    )

    wide.to_csv(
        os.path.join(TABLE_DIR, "best_epoch_results_wide.csv"),
        index=False,
    )

    # LaTeX table
    latex_df = wide.copy()

    for col in latex_df.columns:
        if col.startswith("macro_f1"):
            latex_df[col] = latex_df[col].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
        if col.startswith("epoch"):
            latex_df[col] = latex_df[col].map(lambda x: f"{int(x)}" if pd.notna(x) else "")

    latex_path = os.path.join(TABLE_DIR, "best_epoch_results_wide.tex")
    with open(latex_path, "w") as f:
        f.write(
            latex_df.to_latex(
                index=False,
                escape=False,
            )
        )

    return best, wide


best_long_df, best_wide_df = make_best_result_table(metrics_df)

print("\nBest results, wide:")
print(best_wide_df)


# ============================================================
# Plots
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_macro_f1_by_epoch_per_dataset(metrics_df):
    df = metrics_df[metrics_df["dataset"].isin(TARGET_DATASETS)].copy()

    for dataset in TARGET_DATASETS:
        sub = df[df["dataset"] == dataset].copy()

        if sub.empty:
            print(f"[Skip] No data for {dataset}")
            continue

        plt.figure(figsize=(7.5, 4.8))

        sns.lineplot(
            data=sub,
            x="epoch",
            y="macro_f1",
            hue="model_label",
            marker="o",
            linewidth=2,
            errorbar=None,
        )

        plt.xlabel("Epoch")
        plt.ylabel("Macro-F1")
        plt.title(dataset)
        plt.legend(
            title="Model",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
        )

        savefig(os.path.join(OUTPUT_DIR, f"macro_f1_by_epoch_{dataset}.png"))


def plot_best_macro_f1_bar(best_long_df):
    df = best_long_df.copy()
    df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

    plt.figure(figsize=(9.5, 4.8))

    sns.barplot(
        data=df,
        x="dataset",
        y="macro_f1",
        hue="model_label",
    )

    plt.xlabel("Dataset")
    plt.ylabel("Best Macro-F1")
    plt.title("Best validation/external performance")
    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    savefig(os.path.join(OUTPUT_DIR, "best_macro_f1_all_datasets.png"))


def plot_best_epoch_bar(best_long_df):
    df = best_long_df.copy()
    df = df[df["dataset"].isin(TARGET_DATASETS)].copy()

    plt.figure(figsize=(9.5, 4.8))

    sns.barplot(
        data=df,
        x="dataset",
        y="epoch",
        hue="model_label",
    )

    plt.xlabel("Dataset")
    plt.ylabel("Best epoch")
    plt.title("Epoch selected by best Macro-F1")
    plt.legend(
        title="Model",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )

    savefig(os.path.join(OUTPUT_DIR, "best_epoch_all_datasets.png"))


def plot_main_val_vs_external(best_wide_df):
    df = best_wide_df.copy()

    if "macro_f1_main_val" not in df.columns:
        print("[Skip] No main_val results.")
        return

    external_cols = [
        c for c in [
            "macro_f1_xed_en",
            "macro_f1_emoevent_en",
        ]
        if c in df.columns
    ]

    if not external_cols:
        print("[Skip] No external dataset results.")
        return

    df["external_mean_macro_f1"] = df[external_cols].mean(axis=1)

    plt.figure(figsize=(5.8, 4.8))

    sns.scatterplot(
        data=df,
        x="macro_f1_main_val",
        y="external_mean_macro_f1",
        hue="model_label",
        s=80,
    )

    for _, row in df.iterrows():
        plt.text(
            row["macro_f1_main_val"] + 0.002,
            row["external_mean_macro_f1"] + 0.002,
            row["model_label"],
            fontsize=8,
        )

    plt.xlabel("Best Macro-F1 on main val")
    plt.ylabel("Mean best Macro-F1 on external datasets")
    plt.title("Main validation vs. external generalization")
    plt.legend([], [], frameon=False)

    savefig(os.path.join(OUTPUT_DIR, "main_val_vs_external_mean_macro_f1.png"))


plot_macro_f1_by_epoch_per_dataset(metrics_df)
plot_best_macro_f1_bar(best_long_df)
plot_best_epoch_bar(best_long_df)
plot_main_val_vs_external(best_wide_df)

print("\nSaved plots to:", OUTPUT_DIR)
print("Saved tables to:", TABLE_DIR)

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.multitest import multipletests


# ============================================================
# CONFIG
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

GEMMA3_SMALL_MODELS = {
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
}

BASE_DATA_DIR = "data"
LAMBDA_L1 = 0.0

FEW_SHOT_RESULT_DIR = "data/few_shot_results"

OUTPUT_DIR = "data/statistical_tests"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# PATH HELPERS
# ============================================================

def model_safe_name(model_name: str) -> str:
    return model_name.replace("/", "-")


def display_model_name(model_name: str) -> str:
    name_map = {
        "google/gemma-2-2b": "Gemma-2-2B",
        "google/gemma-2-9b": "Gemma-2-9B",
        "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
        "google/gemma-3-4b-it": "Gemma-3-4B-IT",
        "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
        "google/gemma-3-12b-it": "Gemma-3-12B-IT",
        "meta-llama/Llama-3.1-8B": "Llama-3.1-8B",
    }
    return name_map.get(model_name, model_name)


def get_sae_size(model_name: str):
    if model_name in GEMMA3_SMALL_MODELS:
        return "small"
    return None


def get_feature_dir(model_name: str):
    slug = model_safe_name(model_name)
    sae_size = get_sae_size(model_name)

    if sae_size == "small":
        return os.path.join(BASE_DATA_DIR, f"{slug}-feature-act-small")

    return os.path.join(BASE_DATA_DIR, f"{slug}-feature-act")


def get_steering_output_dir(model_name: str, lambda_l1: float = 0.0):
    feature_dir = get_feature_dir(model_name)
    l1_tag = str(lambda_l1).replace(".", "p")

    # This follows your feature_steering.py get_output_dir().
    return os.path.join(
        feature_dir,
        "global_feature_steering",
        f"lr_5e-3_l1_{l1_tag}",
    )


def normalize_dataset_name(x: str) -> str:
    x = str(x).lower().strip()

    if x in {"main", "main_test", "main_full", "self_reported", "self-reported"}:
        return "main"

    if x in {"main_val", "val", "validation"}:
        return "main_val"

    if x in {"emoevent", "emoevent_en", "emo_event", "emo_event_en"}:
        return "emoevent"

    if x in {"xed", "xed_en", "xed-en"}:
        return "xed"

    return x


# ============================================================
# LOAD STEERING RESULTS
# ============================================================

def choose_best_steering_epoch(metrics_df: pd.DataFrame) -> int:
    """
    Choose the best steering epoch using main_val macro-F1.
    If main_val does not exist, use main macro-F1.
    Epoch 0 is baseline, so we choose from epoch > 0.
    """

    df = metrics_df.copy()
    df["dataset_norm"] = df["dataset"].apply(normalize_dataset_name)

    candidate = df[
        (df["epoch"] > 0)
        & (df["dataset_norm"] == "main_val")
    ].copy()

    if len(candidate) == 0:
        candidate = df[
            (df["epoch"] > 0)
            & (df["dataset_norm"] == "main")
        ].copy()

    if len(candidate) == 0:
        raise ValueError("Cannot find any epoch > 0 row for main_val or main.")

    best_row = candidate.sort_values(
        ["macro_f1", "epoch"],
        ascending=[False, True],
    ).iloc[0]

    return int(best_row["epoch"])


def load_one_model_steering(model_name: str):
    output_dir = get_steering_output_dir(model_name, LAMBDA_L1)

    metrics_path = os.path.join(output_dir, "epoch_metrics_long.csv")
    per_emotion_path = os.path.join(output_dir, "epoch_per_emotion_f1.csv")

    if not os.path.exists(metrics_path):
        warnings.warn(f"Missing steering metrics: {metrics_path}")
        return []

    if not os.path.exists(per_emotion_path):
        warnings.warn(f"Missing steering per-emotion file: {per_emotion_path}")
        return []

    metrics_df = pd.read_csv(metrics_path)
    per_emotion_df = pd.read_csv(per_emotion_path)

    metrics_df["dataset_norm"] = metrics_df["dataset"].apply(normalize_dataset_name)
    per_emotion_df["dataset_norm"] = per_emotion_df["dataset"].apply(normalize_dataset_name)

    best_epoch = choose_best_steering_epoch(metrics_df)

    rows = []

    # -------------------------
    # Baseline = epoch 0
    # -------------------------
    baseline_metrics = metrics_df[metrics_df["epoch"] == 0].copy()
    baseline_per_emotion = per_emotion_df[per_emotion_df["epoch"] == 0].copy()

    rows.extend(
        build_metric_rows_from_steering_dfs(
            model_name=model_name,
            setting="baseline",
            metrics_df=baseline_metrics,
            per_emotion_df=baseline_per_emotion,
            selected_epoch=0,
        )
    )

    # -------------------------
    # Steered = best epoch
    # -------------------------
    steered_metrics = metrics_df[metrics_df["epoch"] == best_epoch].copy()
    steered_per_emotion = per_emotion_df[per_emotion_df["epoch"] == best_epoch].copy()

    rows.extend(
        build_metric_rows_from_steering_dfs(
            model_name=model_name,
            setting="steering",
            metrics_df=steered_metrics,
            per_emotion_df=steered_per_emotion,
            selected_epoch=best_epoch,
        )
    )

    return rows


def build_metric_rows_from_steering_dfs(
    model_name: str,
    setting: str,
    metrics_df: pd.DataFrame,
    per_emotion_df: pd.DataFrame,
    selected_epoch: int,
):
    rows = []

    for dataset_norm in ["main", "emoevent", "xed"]:
        d = metrics_df[metrics_df["dataset_norm"] == dataset_norm].copy()

        if len(d) > 0:
            row = d.iloc[0]

            rows.append({
                "model_name": model_name,
                "model": display_model_name(model_name),
                "setting": setting,
                "dataset": dataset_norm,
                "metric": "macro_f1",
                "value": float(row["macro_f1"]),
                "epoch": selected_epoch,
            })

            if "weighted_f1" in row:
                rows.append({
                    "model_name": model_name,
                    "model": display_model_name(model_name),
                    "setting": setting,
                    "dataset": dataset_norm,
                    "metric": "weighted_f1",
                    "value": float(row["weighted_f1"]),
                    "epoch": selected_epoch,
                })

            if "accuracy" in row:
                rows.append({
                    "model_name": model_name,
                    "model": display_model_name(model_name),
                    "setting": setting,
                    "dataset": dataset_norm,
                    "metric": "accuracy",
                    "value": float(row["accuracy"]),
                    "epoch": selected_epoch,
                })

            if "perplexity" in row and pd.notna(row["perplexity"]):
                rows.append({
                    "model_name": model_name,
                    "model": display_model_name(model_name),
                    "setting": setting,
                    "dataset": dataset_norm,
                    "metric": "perplexity",
                    "value": float(row["perplexity"]),
                    "epoch": selected_epoch,
                })

            if "general_perplexity" in row and pd.notna(row["general_perplexity"]):
                rows.append({
                    "model_name": model_name,
                    "model": display_model_name(model_name),
                    "setting": setting,
                    "dataset": "wikitext",
                    "metric": "general_perplexity",
                    "value": float(row["general_perplexity"]),
                    "epoch": selected_epoch,
                })

        pe = per_emotion_df[per_emotion_df["dataset_norm"] == dataset_norm].copy()

        if len(pe) > 0:
            for emotion in EMOTIONS:
                emo_row = pe[pe["emotion"].astype(str).str.lower() == emotion]

                if len(emo_row) > 0:
                    rows.append({
                        "model_name": model_name,
                        "model": display_model_name(model_name),
                        "setting": setting,
                        "dataset": dataset_norm,
                        "metric": f"{emotion}_f1",
                        "value": float(emo_row.iloc[0]["f1"]),
                        "epoch": selected_epoch,
                    })

    return rows


# ============================================================
# LOAD FEW-SHOT RESULTS
# ============================================================

def load_one_model_fewshot(model_name: str):
    safe_name = model_safe_name(model_name)
    model_dir = os.path.join(FEW_SHOT_RESULT_DIR, safe_name)

    summary_path = os.path.join(model_dir, "all_dataset_summary.csv")
    per_emotion_path = os.path.join(model_dir, "all_dataset_per_emotion_f1.csv")

    if not os.path.exists(summary_path):
        warnings.warn(f"Missing few-shot summary: {summary_path}")
        return []

    if not os.path.exists(per_emotion_path):
        warnings.warn(f"Missing few-shot per-emotion file: {per_emotion_path}")
        return []

    summary_df = pd.read_csv(summary_path)
    per_emotion_df = pd.read_csv(per_emotion_path)

    summary_df["dataset_norm"] = summary_df["dataset"].apply(normalize_dataset_name)
    per_emotion_df["dataset_norm"] = per_emotion_df["dataset"].apply(normalize_dataset_name)

    rows = []

    for dataset_norm in ["main", "emoevent", "xed"]:
        d = summary_df[summary_df["dataset_norm"] == dataset_norm].copy()

        if len(d) > 0:
            row = d.iloc[0]

            rows.append({
                "model_name": model_name,
                "model": display_model_name(model_name),
                "setting": "fewshot",
                "dataset": dataset_norm,
                "metric": "macro_f1",
                "value": float(row["macro_f1"]),
                "epoch": np.nan,
            })

            if "weighted_f1" in row:
                rows.append({
                    "model_name": model_name,
                    "model": display_model_name(model_name),
                    "setting": "fewshot",
                    "dataset": dataset_norm,
                    "metric": "weighted_f1",
                    "value": float(row["weighted_f1"]),
                    "epoch": np.nan,
                })

            if "accuracy" in row:
                rows.append({
                    "model_name": model_name,
                    "model": display_model_name(model_name),
                    "setting": "fewshot",
                    "dataset": dataset_norm,
                    "metric": "accuracy",
                    "value": float(row["accuracy"]),
                    "epoch": np.nan,
                })

        pe = per_emotion_df[per_emotion_df["dataset_norm"] == dataset_norm].copy()

        if len(pe) > 0:
            for emotion in EMOTIONS:
                emo_row = pe[pe["emotion"].astype(str).str.lower() == emotion]

                if len(emo_row) > 0:
                    rows.append({
                        "model_name": model_name,
                        "model": display_model_name(model_name),
                        "setting": "fewshot",
                        "dataset": dataset_norm,
                        "metric": f"{emotion}_f1",
                        "value": float(emo_row.iloc[0]["f1"]),
                        "epoch": np.nan,
                    })

    return rows


# ============================================================
# STATISTICAL TESTS
# ============================================================

def cohen_dz(x, y):
    diff = y - x
    sd = np.std(diff, ddof=1)

    if sd == 0 or np.isnan(sd):
        return np.nan

    return float(np.mean(diff) / sd)


def bootstrap_mean_diff_ci(x, y, n_boot=10000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    diff = y - x

    boot_means = []
    n = len(diff)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means.append(np.mean(diff[idx]))

    alpha = (100 - ci) / 2
    low = np.percentile(boot_means, alpha)
    high = np.percentile(boot_means, 100 - alpha)

    return float(low), float(high)


def safe_wilcoxon(x, y, alternative="greater"):
    diff = y - x

    if len(diff) < 2:
        return np.nan, np.nan

    if np.allclose(diff, 0):
        return 0.0, 1.0

    try:
        stat, p = wilcoxon(
            y,
            x,
            alternative=alternative,
            zero_method="wilcox",
            correction=False,
            mode="auto",
        )
        return float(stat), float(p)
    except Exception as e:
        warnings.warn(f"Wilcoxon failed: {repr(e)}")
        return np.nan, np.nan


def safe_ttest(x, y, alternative="greater"):
    diff = y - x

    if len(diff) < 2:
        return np.nan, np.nan

    if np.allclose(diff, 0):
        return 0.0, 1.0

    try:
        stat, p = ttest_rel(y, x, alternative=alternative)
        return float(stat), float(p)
    except Exception as e:
        warnings.warn(f"Paired t-test failed: {repr(e)}")
        return np.nan, np.nan


def run_one_comparison(
    df: pd.DataFrame,
    setting_a: str,
    setting_b: str,
    comparison_name: str,
    dataset: str,
    metric: str,
    alternative: str = "greater",
):
    a = df[
        (df["setting"] == setting_a)
        & (df["dataset"] == dataset)
        & (df["metric"] == metric)
    ][["model_name", "value"]].rename(columns={"value": "a"})

    b = df[
        (df["setting"] == setting_b)
        & (df["dataset"] == dataset)
        & (df["metric"] == metric)
    ][["model_name", "value"]].rename(columns={"value": "b"})

    merged = a.merge(b, on="model_name", how="inner")

    if len(merged) < 2:
        return None

    x = merged["a"].to_numpy(dtype=float)
    y = merged["b"].to_numpy(dtype=float)

    w_stat, w_p = safe_wilcoxon(x, y, alternative=alternative)
    t_stat, t_p = safe_ttest(x, y, alternative=alternative)
    ci_low, ci_high = bootstrap_mean_diff_ci(x, y)

    out = {
        "comparison": comparison_name,
        "setting_a": setting_a,
        "setting_b": setting_b,
        "dataset": dataset,
        "metric": metric,
        "alternative": alternative,
        "n_models": int(len(merged)),
        "models": "; ".join(merged["model_name"].tolist()),
        "mean_a": float(np.mean(x)),
        "mean_b": float(np.mean(y)),
        "mean_diff_b_minus_a": float(np.mean(y - x)),
        "median_diff_b_minus_a": float(np.median(y - x)),
        "relative_change_pct": float((np.mean(y) - np.mean(x)) / np.mean(x) * 100) if np.mean(x) != 0 else np.nan,
        "cohen_dz": cohen_dz(x, y),
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "wilcoxon_stat": w_stat,
        "wilcoxon_p": w_p,
        "paired_t_stat": t_stat,
        "paired_t_p": t_p,
    }

    return out


def run_all_tests(combined_df: pd.DataFrame):
    comparisons = [
        ("baseline", "steering", "steering_vs_baseline"),
        ("baseline", "fewshot", "fewshot_vs_baseline"),
        ("fewshot", "steering", "steering_vs_fewshot"),
    ]

    datasets = ["main", "emoevent", "xed"]

    metrics = [
        "macro_f1",
        "weighted_f1",
        "accuracy",
        "anger_f1",
        "disgust_f1",
        "fear_f1",
        "joy_f1",
        "sadness_f1",
        "surprise_f1",
    ]

    rows = []

    for setting_a, setting_b, comparison_name in comparisons:
        for dataset in datasets:
            for metric in metrics:
                row = run_one_comparison(
                    df=combined_df,
                    setting_a=setting_a,
                    setting_b=setting_b,
                    comparison_name=comparison_name,
                    dataset=dataset,
                    metric=metric,
                    alternative="greater",
                )

                if row is not None:
                    rows.append(row)

    tests_df = pd.DataFrame(rows)

    if len(tests_df) > 0:
        valid = tests_df["wilcoxon_p"].notna()

        tests_df["wilcoxon_p_holm"] = np.nan
        tests_df["wilcoxon_reject_holm_0.05"] = False

        if valid.sum() > 0:
            reject, p_adj, _, _ = multipletests(
                tests_df.loc[valid, "wilcoxon_p"].values,
                alpha=0.05,
                method="holm",
            )

            tests_df.loc[valid, "wilcoxon_p_holm"] = p_adj
            tests_df.loc[valid, "wilcoxon_reject_holm_0.05"] = reject

    return tests_df


# ============================================================
# TABLE FORMAT FOR PAPER
# ============================================================

def make_wide_table(combined_df: pd.DataFrame):
    keep_metrics = [
        "macro_f1",
        "anger_f1",
        "disgust_f1",
        "fear_f1",
        "joy_f1",
        "sadness_f1",
        "surprise_f1",
    ]

    df = combined_df[
        (combined_df["dataset"].isin(["main", "emoevent", "xed"]))
        & (combined_df["metric"].isin(keep_metrics))
    ].copy()

    wide = df.pivot_table(
        index=["model", "model_name", "setting"],
        columns=["dataset", "metric"],
        values="value",
        aggfunc="first",
    )

    wide.columns = [
        f"{dataset}_{metric}"
        for dataset, metric in wide.columns
    ]

    wide = wide.reset_index()

    return wide


# ============================================================
# MAIN
# ============================================================

def main():
    all_rows = []

    print("Loading steering baseline and steered results...")
    for model_name in MODELS:
        rows = load_one_model_steering(model_name)
        all_rows.extend(rows)

    print("Loading few-shot results...")
    for model_name in MODELS:
        rows = load_one_model_fewshot(model_name)
        all_rows.extend(rows)

    combined_df = pd.DataFrame(all_rows)

    if len(combined_df) == 0:
        raise RuntimeError("No results were loaded. Check your paths.")

    combined_df = combined_df.sort_values(
        ["model_name", "setting", "dataset", "metric"]
    ).reset_index(drop=True)

    combined_path = os.path.join(OUTPUT_DIR, "combined_metrics_long.csv")
    combined_df.to_csv(combined_path, index=False)

    wide_df = make_wide_table(combined_df)
    wide_path = os.path.join(OUTPUT_DIR, "combined_metrics_wide.csv")
    wide_df.to_csv(wide_path, index=False)

    print("Running paired statistical tests...")
    tests_df = run_all_tests(combined_df)

    tests_path = os.path.join(OUTPUT_DIR, "paired_statistical_tests.csv")
    tests_df.to_csv(tests_path, index=False)

    # A smaller table for the paper-relevant claims.
    paper_focus = tests_df[
        (
            tests_df["comparison"].isin([
                "steering_vs_baseline",
                "fewshot_vs_baseline",
                "steering_vs_fewshot",
            ])
        )
        & (
            (
                tests_df["dataset"].isin(["main", "emoevent", "xed"])
                & (tests_df["metric"] == "macro_f1")
            )
            |
            (
                (tests_df["dataset"] == "main")
                & (tests_df["metric"].isin(["disgust_f1", "surprise_f1"]))
            )
        )
    ].copy()

    paper_focus_path = os.path.join(OUTPUT_DIR, "paper_focus_statistical_tests.csv")
    paper_focus.to_csv(paper_focus_path, index=False)

    print("\nSaved:")
    print(f"  {combined_path}")
    print(f"  {wide_path}")
    print(f"  {tests_path}")
    print(f"  {paper_focus_path}")

    print("\nPaper-focused tests:")
    with pd.option_context("display.max_columns", None, "display.width", 200):
        print(
            paper_focus[
                [
                    "comparison",
                    "dataset",
                    "metric",
                    "n_models",
                    "mean_a",
                    "mean_b",
                    "mean_diff_b_minus_a",
                    "relative_change_pct",
                    "wilcoxon_p",
                    "wilcoxon_p_holm",
                    "cohen_dz",
                    "bootstrap_ci_low",
                    "bootstrap_ci_high",
                ]
            ].round(4)
        )


if __name__ == "__main__":
    main()